# Webapp Backend Smoke Test (§5.4)

Real requests against `webapp/backend/main.py` via FastAPI's `TestClient` -- exercises the actual route handlers (no mocked HTTP), the same way `4.1-4.2 speech smoke test.ipynb` and Streamlit's `AppTest` verify their layers for real rather than assumed.

**Cost:** mock-mode endpoints are $0. Parts C-E make real calls (1 real ASR transcription -- free, local; 1 real end-to-end graph query -- a few real LLM calls; 1 real TTS call) so the FastAPI-to-graph wiring is verified for real, not just in mock mode. Totals in the cost-summary cell at the end.

## Setup

In [ ]:
import json
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "webapp", "backend"))

from fastapi.testclient import TestClient

import main as backend  # webapp/backend/main.py

client = TestClient(backend.app)
print("TestClient ready. Routes:", [r.path for r in backend.app.routes if r.path.startswith("/api")])

## Part A: `/api/config` and `/api/prompt-log` ($0 -- no LLM/tool calls)

In [ ]:
resp = client.get("/api/config")
assert resp.status_code == 200, resp.text
config = resp.json()
print(config)
assert set(config) == {"llm_provider", "llm_model", "llm_configured", "asr", "tts", "mock_examples"}
assert len(config["mock_examples"]) == 4  # matches ui_common.py's MOCK_EXAMPLES (added an out_of_scope example)

resp = client.get("/api/prompt-log")
assert resp.status_code == 200, resp.text
print("prompt-log entries so far:", len(resp.json()))

## Part B: `/api/query` in mock mode ($0)

Streams NDJSON. For each of the 4 canned examples (`ui_common.MOCK_EXAMPLES` -- budget search, current-price search, safety-flag refusal, out-of-scope redirect), checks that every step lands and the final payload matches the canned data exactly -- plus the unknown-query error case.

In [ ]:
def stream_query(query: str, mock: bool):
    resp = client.post("/api/query", json={"query": query, "mock": mock})
    assert resp.status_code == 200, resp.text
    lines = [json.loads(line) for line in resp.text.strip().split("\n") if line.strip()]
    return lines


for query, example in backend.MOCK_EXAMPLES.items():
    events = stream_query(query, mock=True)
    steps, final = events[:-1], events[-1]
    assert [s["node"] for s in steps] == [node for node, _ in example["step_log"]], (query, events)
    assert final["type"] == "final"
    assert final["final_answer"] == example["final_answer"]
    assert final["products"] == example["products"]
    assert final["citations"] == example["citations"]
    print(f"OK -- {query!r}: {len(steps)} step(s), final_answer matches canned data")

# Unknown query -- exact-match miss, expect a typed error event, not a crash
events = stream_query("not a real mock example", mock=True)
assert events[0]["type"] == "error"
print("OK -- unknown mock query returns a typed error event:", events[0]["message"])

## Part C: `/api/transcribe` (real, local faster-whisper for the transcription itself -- $0; 1 real TTS call to synthesize the input clip)

Same pattern as `4.3 performance and reliability.ipynb`'s Part A: a real TTS-synthesized clip stands in for a mic recording, so the round trip through `UploadFile` -> temp file -> `asr.transcribe()` is exercised for real.

In [ ]:
import io

input_text = "kids storage bins under fifteen dollars"
input_audio = backend.tts.synthesize(input_text)  # real TTS call -- stands in for a mic recording

resp = client.post(
    "/api/transcribe",
    files={"file": ("input.wav", io.BytesIO(input_audio.audio_bytes), "audio/wav")},
)
assert resp.status_code == 200, resp.text
transcribed = resp.json()
print(transcribed)
assert transcribed["text"].strip(), "expected a non-empty transcript"

## Part D: `/api/query` in real mode (real -- a few real LLM calls through the actual graph)

Confirms the FastAPI layer's `_stream_real()` correctly drives `graph.astream(..., stream_mode="updates")` end to end -- same graph §2.7's testing notebook already validated, but this is the first real check that the *webapp's* HTTP layer wires to it correctly (streaming NDJSON, per-step summaries, final payload shape).

In [ ]:
events = stream_query("kids storage bins under $15", mock=False)
steps, final = events[:-1], events[-1]

print("Nodes streamed:", [s["node"] for s in steps])
for s in steps:
    print(f"  {s['node']}: {s['summary']}")

assert final["type"] == "final"
# "query" (the Router's cleaned-up search query, not the raw transcript) was
# added after a real UI test showed the frontend's follow-up chips were
# templating onto the raw transcript instead -- broken for a full-sentence
# query. See main.py's _final_payload() docstring.
assert set(final) == {"type", "products", "citations", "final_answer", "errors", "query"}
print("\nfinal_answer:", final["final_answer"])
print("query (router-cleaned):", final["query"])
print("products:", len(final["products"]), "| citations:", len(final["citations"]), "| errors:", final["errors"])

## Part E: `/api/tts` (real -- 1 more real TTS call)

In [ ]:
resp = client.post("/api/tts", json={"text": final["final_answer"] or "This is a smoke test."})
assert resp.status_code == 200, resp.text
assert resp.headers["content-type"] == "audio/wav"
audio_bytes = resp.content
print(f"OK -- {len(audio_bytes)} bytes of real WAV audio returned")

# Empty text is a real TTSError -- confirms it's surfaced as 503, not a 500 crash
resp = client.post("/api/tts", json={"text": ""})
print("empty-text status:", resp.status_code, resp.json())
assert resp.status_code == 503

## Cost summary

In [ ]:
from collections import Counter

llm_calls = Counter(c["node"] for c in backend.providers.LLM_CALL_LOG)
print(f"Real LLM calls this notebook: {sum(llm_calls.values())} -- {dict(llm_calls)}")
print("Real TTS calls this notebook: 2 (1 input-synth fixture in Part C, 1 real answer in Part E)")
print("Note: mcp_client.get_session() is now guarded by an asyncio.Lock (fixed after this notebook's "
      "first draft -- see mcp_client.py's module docstring), closing the concurrent-first-query race a "
      "real deployed backend could hit. Still open, tracked separately: providers.LLM_CALL_LOG is one "
      "process-global list with no per-session scoping, and GET /api/prompt-log exposes it directly -- "
      "fine for a local single-user demo, real cross-session exposure if this backend is ever shared.")